# Laboratorul 6 — RAG pentru documentația oficială Python

**Autor: Carp Dan**  
**Tema:** asistent de întrebări și răspunsuri pentru concepte Python, cu recuperare de fragmente și citarea surselor.

Acest laborator este o lucrare nouă, cu un corpus diferit de exemplele de referință. Baza de cunoștințe este un set didactic scurt, formulat prin parafrazarea documentației oficiale Python. Notebook-ul nu conține rezultate simulate: celulele sunt pregătite pentru rulare, iar rezultatele vor apărea numai după executarea lor de către student.

## Obiective

La finalul laboratorului, studentul va putea:

1. să construiască și să segmenteze o bază de cunoștințe cu metadate și surse;
2. să compare BM25, embeddings, FAISS, RRF și reranking;
3. să măsoare Hit@k, KW@k, MRR, latența și efectul pragului de similaritate;
4. să compare generarea fără context cu generarea augmentată;
5. să evalueze acuratețea, abținerea, citarea și latența, apoi să construiască asistentul final.

RAG nu garantează adevărul răspunsului. Sistemul trebuie să indice sursele recuperate, să se abțină când corpusul nu conține răspunsul și să fie verificat de o persoană.

## 0. Pregătirea mediului

Modelul de embeddings și modelul generativ se descarcă de pe Hugging Face la prima utilizare și rulează local după descărcare; este necesară conexiune la internet doar pentru descărcarea inițială. Dimensiunea dependențelor și timpul de descărcare depind de mediu. Nu este necesară o cheie API. FAISS este opțional: îl poți instala separat cu `%pip install faiss-cpu` dacă există un wheel compatibil cu sistemul tău; altfel, căutarea NumPy oferă fallback-ul.

Rulează celula de instalare o singură dată în kernelul notebook-ului, apoi repornește kernelul dacă mediul o cere.

In [ ]:
%pip install -q sentence-transformers transformers torch sentencepiece pypdf python-docx matplotlib


In [ ]:
from collections import Counter
import csv
import json
import math
import platform
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline


## 1. Formularea problemei

Vom construi un asistent pentru întrebări introductive despre sintaxa și conceptele Python. Corpusul conține 19 înregistrări scurte, fiecare cu identificator, titlu, text parafrazat și URL către capitolul oficial relevant. Înregistrările sunt exemple didactice curate, nu o descărcare integrală a documentației.

Întrebările de evaluare sunt scrise separat de corpus. Pentru fiecare întrebare răspunsabilă se notează documentul relevant, ceea ce permite evaluarea recuperării fără a confunda calitatea căutării cu calitatea textului generat.

### Surse primare

Toate sursele de mai jos sunt pagini din tutorialul oficial Python 3, consultate la 8 noiembrie 2026. Conținutul din corpus este parafrazat și redus pentru exercițiu; URL-urile permit verificarea independentă.

| Subiecte din corpus | Sursă |
|---|---|
| Liste, comprehensions, tupluri și secvențe | [Data Structures](https://docs.python.org/3/tutorial/datastructures.html) |
| Flux de control și condiții | [More Control Flow Tools](https://docs.python.org/3/tutorial/controlflow.html) |
| Funcții și parametri | [Defining Functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions) |
| Dicționare și mulțimi | [Data Structures](https://docs.python.org/3/tutorial/datastructures.html) |
| Erori și excepții | [Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html) |
| Fișiere și formatarea ieșirii | [Input and Output](https://docs.python.org/3/tutorial/inputoutput.html) |
| Module și pachete | [Modules](https://docs.python.org/3/tutorial/modules.html) |
| Clase și moștenire | [Classes](https://docs.python.org/3/tutorial/classes.html) |
| Iteratori și generatoare | [Iterators](https://docs.python.org/3/tutorial/classes.html#iterators) |

Documentația este actualizată în timp; verifică versiunea și data paginii dacă folosești răspunsurile în afara acestui laborator.

## 2. Baza de cunoștințe

Textele sunt scurte rezumate proprii ale surselor primare. Păstrăm metadatele lângă text, astfel încât răspunsul final să poată indica titlul și URL-ul documentului, nu doar fragmentul.

In [ ]:
documents = [
    {
        'id': 'lists',
        'title': 'Lists: mutation and removing elements',
        'source': 'https://docs.python.org/3/tutorial/datastructures.html',
        'text': (
            'Lists are mutable sequences. append adds an item at the end. remove deletes '
            'the first item equal to the requested value and raises ValueError if the '
            'value is absent. pop removes and returns the item at the given index; '
            'without an index, pop removes the last item. pop raises IndexError if the '
            'list is empty or the index is outside the list.'
        ),
    },
    {
        'id': 'dictionaries',
        'title': 'Dictionaries: keys and retrieving values',
        'source': 'https://docs.python.org/3/tutorial/datastructures.html',
        'text': (
            'A dictionary maps keys to values. Keys are unique; assigning a value to an '
            'existing key replaces its previous value. The get(key, default) method '
            'returns the default when a key is absent instead of raising KeyError.'
        ),
    },
    {
        'id': 'exceptions',
        'title': 'Syntax errors and exceptions',
        'source': 'https://docs.python.org/3/tutorial/errors.html',
        'text': (
            'Syntax errors are detected while parsing a program. Exceptions happen while '
            'executing a statement that is syntactically valid. A try block can be '
            'followed by except clauses to handle anticipated exception types. An else '
            'clause associated with try runs when no exception occurred in the try block.'
        ),
    },
    {
        'id': 'files',
        'title': 'Files and the with statement',
        'source': 'https://docs.python.org/3/tutorial/inputoutput.html',
        'text': (
            'open returns a file object. Mode r is for reading, w is for writing and '
            'replaces existing content, and a appends to the end. Using with for a file '
            'object closes it correctly after the block, including when an exception '
            'occurs. For text files, encoding can be specified explicitly in open.'
        ),
    },
    {
        'id': 'modules',
        'title': 'Modules and imports',
        'source': 'https://docs.python.org/3/tutorial/modules.html',
        'text': (
            'A module is a Python file that can contain definitions and statements. '
            'Definitions in a module can be reused by importing it. The statement import '
            'module adds the module name to the current namespace; its functions are '
            'accessed with module.function notation.'
        ),
    },
    {
        'id': 'list-comprehensions',
        'title': 'List comprehensions',
        'source': 'https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions',
        'text': 'A list comprehension builds a list from an expression and one or more for clauses, optionally filtered with if. It is a concise way to create a new list from a sequence.',
    },
    {
        'id': 'tuples',
        'title': 'Tuples and nested sequences',
        'source': 'https://docs.python.org/3/tutorial/datastructures.html#tuples-and-sequences',
        'text': 'Tuples are ordered sequences often used to group values. A one-item tuple is written with a trailing comma. Sequence unpacking assigns the elements to variables in the same order.',
    },
    {
        'id': 'sets',
        'title': 'Sets and set operations',
        'source': 'https://docs.python.org/3/tutorial/datastructures.html#sets',
        'text': 'Sets contain unique elements and do not have a fixed order. They support membership tests and operations such as union, intersection, and difference.',
    },
    {
        'id': 'if-statements',
        'title': 'Conditional statements',
        'source': 'https://docs.python.org/3/tutorial/controlflow.html#if-statements',
        'text': 'An if statement executes the branch whose test is true. elif allows alternative tests, and else defines the default branch. Blocks are delimited by indentation.',
    },
    {
        'id': 'for-loops',
        'title': 'Iteration with for and range',
        'source': 'https://docs.python.org/3/tutorial/controlflow.html#for-statements',
        'text': 'A for statement iterates over the items of a sequence or other iterable. range produces an arithmetic progression of values and is commonly used when a block must repeat a specific number of times.',
    },
    {
        'id': 'functions',
        'title': 'Defining functions and return',
        'source': 'https://docs.python.org/3/tutorial/controlflow.html#defining-functions',
        'text': 'Functions are defined with def, followed by a name and parameters. return ends the call and can provide a value; if omitted, the function returns None. A docstring can document the function purpose.',
    },
    {
        'id': 'function-parameters',
        'title': 'Default parameters and arguments',
        'source': 'https://docs.python.org/3/tutorial/controlflow.html#default-argument-values',
        'text': 'A parameter can have a default value, used when the caller omits its argument. Keyword arguments bind values to parameter names. Default values are evaluated once when the function is defined, so mutable objects require care.',
    },
    {
        'id': 'file-formats',
        'title': 'Reading text files and encoding',
        'source': 'https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files',
        'text': 'Text files are read and written as strings using a character encoding. Specifying an encoding such as UTF-8 avoids relying on the system default. Binary files are opened with a mode that includes b.',
    },
    {
        'id': 'formatting',
        'title': 'Formatting strings',
        'source': 'https://docs.python.org/3/tutorial/inputoutput.html#formatted-string-literals',
        'text': 'F-strings allow expressions in braces inside a string literal prefixed with f. Format specifiers can control how values are represented, such as the number of decimal places for a floating-point number.',
    },
    {
        'id': 'try-finally',
        'title': 'Resource cleanup with finally',
        'source': 'https://docs.python.org/3/tutorial/errors.html#defining-clean-up-actions',
        'text': 'The finally clause runs when leaving a try statement, whether or not an exception occurred. It is intended for cleanup actions that should be attempted in all cases; for files, with is usually simpler.',
    },
    {
        'id': 'classes',
        'title': 'Classes, instances, and methods',
        'source': 'https://docs.python.org/3/tutorial/classes.html',
        'text': 'A class defines attributes and methods for objects created as its instances. Instance methods receive the current instance, conventionally named self. __init__ initializes instance attributes when an object is created.',
    },
    {
        'id': 'inheritance',
        'title': 'Inheritance and method overriding',
        'source': 'https://docs.python.org/3/tutorial/classes.html#inheritance',
        'text': 'A derived class can inherit attributes and methods from a base class and define a method with the same name to change its behavior. super() provides convenient access to the base-class implementation.',
    },
    {
        'id': 'iterators',
        'title': 'Iterables and iterators',
        'source': 'https://docs.python.org/3/tutorial/classes.html#iterators',
        'text': 'An iterable can provide an iterator, and the iterator yields items one at a time. A for statement uses the iterator protocol. An iterator remembers its current position and is consumed as it is traversed.',
    },
    {
        'id': 'generators',
        'title': 'Generator functions and yield',
        'source': 'https://docs.python.org/3/tutorial/classes.html#generators',
        'text': 'A function that uses yield produces a generator. At each step, the generator provides a value and suspends its state, continuing where it left off at the next request. This can avoid building an entire list in advance.',
    },
]

documents_by_id = {document['id']: document for document in documents}
assert len(documents) == len(documents_by_id) == 19
print(f'Documente în corpus: {len(documents)}')
print('ID-uri:', ', '.join(documents_by_id))


## 3. Încărcarea documentelor

Corpusul de bază este inclus direct în notebook pentru reproductibilitate. Pentru experimentul propriu, funcția următoare poate încărca fișiere TXT, CSV, PDF și DOCX dintr-un folder. Păstrează numai documente pe care ai dreptul să le folosești și verifică extragerea PDF-urilor scanate: acestea pot necesita OCR, care nu este implementat aici. Metadatele locale identifică fișierul, dar pentru surse publice completează și URL-ul oficial.

In [ ]:
def load_local_document(path):
    path = Path(path)
    suffix = path.suffix.casefold()
    if suffix == '.txt':
        text = path.read_text(encoding='utf-8')
    elif suffix == '.csv':
        with path.open(encoding='utf-8', newline='') as file:
            rows = csv.DictReader(file)
            text = '\n'.join(' | '.join(str(value or '') for value in row.values()) for row in rows)
    elif suffix == '.pdf':
        from pypdf import PdfReader
        text = '\n'.join(page.extract_text() or '' for page in PdfReader(path).pages)
    elif suffix == '.docx':
        from docx import Document
        text = '\n'.join(paragraph.text for paragraph in Document(path).paragraphs)
    else:
        raise ValueError(f'Format neacceptat: {suffix}')

    text = text.strip()
    if not text:
        raise ValueError(f'Nu s-a extras text din {path}.')
    return {
        'id': path.stem,
        'title': path.name,
        'source': str(path),
        'text': text,
    }


# Exemplu pentru documente proprii; folderul trebuie creat și populat de student:
# local_documents = [load_local_document(path) for path in Path('knowledge_base').glob('*')]


## 4. Segmentarea textului

Modelele de recuperare compară fragmente, nu documente lungi întregi. Funcția următoare împarte fiecare document după cuvinte și păstrează o suprapunere între fragmente vecine. Pentru corpusul mic de aici, unele documente pot produce un singur fragment; pe documente lungi, dimensiunea și suprapunerea trebuie ajustate experimental.

In [ ]:
def make_chunks(documents, chunk_size=55, overlap=10):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError('Este necesar 0 <= overlap < chunk_size.')

    chunks = []
    step = chunk_size - overlap
    for document in documents:
        words = document['text'].split()
        for part, start in enumerate(range(0, len(words), step)):
            text = ' '.join(words[start:start + chunk_size])
            chunks.append({
                'chunk_id': f"{document['id']}-c{part:02d}",
                'doc_id': document['id'],
                'title': document['title'],
                'source': document['source'],
                'text': text,
            })
            if start + chunk_size >= len(words):
                break
    return chunks


CHUNK_SIZE = 55
CHUNK_OVERLAP = 10
chunks = make_chunks(documents, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
print(f'Fragmente create: {len(chunks)}')
for chunk in chunks[:3]:
    print(f"\n{chunk['chunk_id']} [{chunk['title']}]\n{chunk['text']}")


## 5. Indexare semantică prin embeddings

Un embedding reprezintă un text ca vector numeric. Căutarea semantică compară vectorul întrebării cu vectorii fragmentelor prin produs scalar după normalizare, echivalent cu similaritatea cosinus. Modelul de mai jos este orientat spre texte în limba engleză, la fel ca documentația din acest corpus.

La prima rulare se descarcă modelul `sentence-transformers/all-MiniLM-L6-v2`. Nu se descarcă nimic în timpul pregătirii acestui fișier.

In [ ]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
chunk_texts = [chunk['text'] for chunk in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
print('Matrice embeddings:', chunk_embeddings.shape)


## 6. Metode de căutare

BM25 punctează potrivirea termenilor și ia în calcul frecvența termenilor și lungimea documentului. Căutarea semantică compară embeddings normalizate cu produsul scalar. FAISS poate accelera căutarea vectorială la corpusuri mari; pentru acest corpus mic, implementarea NumPy rămâne o alternativă simplă. RRF combină rangurile lexicale și semantice, iar reranking-ul reordonează o listă scurtă de candidați. Comparăm toate variantele pe același set de întrebări.

In [ ]:
# Tokenizare lexicală simplă în engleză; nu include stemming sau eliminarea stop-word-urilor.
def tokenize(text):
    return re.findall(r'[a-z0-9_]+', text.casefold())


tokenized_chunks = [tokenize(chunk['text']) for chunk in chunks]
document_frequency = Counter(
    token for tokens in tokenized_chunks for token in set(tokens)
)
average_length = sum(map(len, tokenized_chunks)) / max(len(tokenized_chunks), 1)


def bm25_search(query, k=3, k1=1.5, b=0.75):
    query_terms = tokenize(query)
    scores = []
    count = len(tokenized_chunks)
    for index, tokens in enumerate(tokenized_chunks):
        term_counts = Counter(tokens)
        length = len(tokens)
        score = 0.0
        for term in query_terms:
            frequency = term_counts[term]
            if frequency == 0:
                continue
            df = document_frequency[term]
            inverse_frequency = math.log(
                1 + (count - df + 0.5) / (df + 0.5)
            )
            denominator = frequency + k1 * (
                1 - b + b * length / max(average_length, 1)
            )
            score += inverse_frequency * frequency * (k1 + 1) / denominator
        scores.append((index, score))
    matched = [(index, score) for index, score in scores if score > 0]
    return sorted(matched, key=lambda item: (-item[1], item[0]))[:k]


def dense_search(query, k=3):
    query_embedding = embedding_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    scores = chunk_embeddings @ query_embedding
    order = np.argsort(-scores)[:k]
    return [(int(index), float(scores[index])) for index in order]


def rrf_search(query, k=3, candidate_k=8, rank_constant=60):
    fused_scores = Counter()
    for results in (
        bm25_search(query, k=candidate_k),
        dense_search(query, k=candidate_k),
    ):
        for rank, (index, _) in enumerate(results, start=1):
            fused_scores[index] += 1 / (rank_constant + rank)
    return sorted(
        fused_scores.items(), key=lambda item: (-item[1], item[0])
    )[:k]


In [ ]:
sample_question = 'What happens when pop is called on a list without an index?'
for method_name, search in (
    ('BM25', bm25_search),
    ('Semantic', dense_search),
    ('Hybrid RRF', rrf_search),
):
    print(f'\n{method_name}')
    for index, score in search(sample_question, k=3):
        chunk = chunks[index]
        print(f"{score:.4f} | {chunk['doc_id']} | {chunk['text']}")


### 6.1 Index FAISS opțional, reranking și selectarea retrieverului

Indexul `IndexFlatIP` caută produsul scalar exact; deoarece embeddings sunt normalizate, scorul echivalează cu similaritatea cosinus. FAISS este opțional: dacă nu este instalat, căutarea NumPy de mai sus rămâne funcțională. Reranking-ul de aici folosește din nou embedding-ul de query pentru a reordona candidații RRF, deci este un baseline semantic ușor, nu un cross-encoder separat.

In [ ]:
try:
    import faiss
except ImportError:
    faiss = None
    faiss_index = None
    print('FAISS nu este instalat; se poate folosi căutarea NumPy.')
else:
    faiss_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    faiss_index.add(np.ascontiguousarray(chunk_embeddings, dtype=np.float32))
    print('Index FAISS creat:', faiss_index.ntotal, 'vectori')


def faiss_search(query, k=3):
    if faiss_index is None:
        return dense_search(query, k=k)
    query_vector = embedding_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)
    scores, indices = faiss_index.search(query_vector, min(k, len(chunks)))
    return [(int(index), float(score)) for index, score in zip(indices[0], scores[0])]


def rerank_search(query, k=3, candidate_k=8):
    candidates = rrf_search(query, k=candidate_k)
    query_vector = embedding_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    rescored = [(index, float(chunk_embeddings[index] @ query_vector))
                for index, _ in candidates]
    return sorted(rescored, key=lambda item: (-item[1], item[0]))[:k]


def make_retriever(mode):
    available = {
        'bm25': bm25_search,
        'semantic_numpy': dense_search,
        'semantic_faiss': faiss_search,
        'rrf': rrf_search,
        'rerank': rerank_search,
    }
    if mode not in available:
        raise ValueError(f'Mod retriever necunoscut: {mode}')
    return available[mode]


retrievers = {mode: make_retriever(mode) for mode in (
    'bm25', 'semantic_numpy', 'semantic_faiss', 'rrf', 'rerank'
)}


### 6.2 Pragul de similaritate și influența lui k

Pragul de scor poate reduce rezultatele nerelevante, dar un prag ales după vizualizarea setului de test poate supraestima performanța. Folosește o parte de validare pentru alegerea pragului și raportează-l explicit. Scorurile cosinus nu sunt probabilități.

In [ ]:
def semantic_threshold_search(query, k=3, threshold=0.30):
    if not 0.0 <= threshold <= 1.0:
        raise ValueError('Pragul cosinus trebuie să fie între 0 și 1.')
    candidates = dense_search(query, k=len(chunks))
    accepted = [(index, score) for index, score in candidates if score >= threshold]
    return accepted[:k]


# Încălzește codarea întrebării înainte de a compara latența metodelor.
dense_search('warm up query', k=1)


## 7. Evaluarea recuperării

Setul de evaluare conține câte o întrebare pentru fiecare document din corpus și este construit manual. El permite o comparație reproductibilă în laborator, dar nu susține concluzii generale despre alte domenii. Hit@k este 1 dacă cel puțin un fragment al documentului așteptat apare în primele k rezultate. Reciprocal Rank (RR) este inversul rangului primului rezultat relevant; MRR este media RR peste întrebări.

In [ ]:
evaluation_questions = [
    {'question': 'What does list.pop return when no index is given?', 'doc_id': 'lists'},
    {'question': 'How can get provide a fallback for a missing dictionary key?', 'doc_id': 'dictionaries'},
    {'question': 'When are syntax errors detected, and when do exceptions happen?', 'doc_id': 'exceptions'},
    {'question': 'Why should a file be opened with a context manager?', 'doc_id': 'files'},
    {'question': 'How do I access a function after importing a module?', 'doc_id': 'modules'},
    {'question': 'How does a list comprehension create a new list?', 'doc_id': 'list-comprehensions'},
    {'question': 'How do I write a tuple containing just one item?', 'doc_id': 'tuples'},
    {'question': 'Which operations can be applied to sets?', 'doc_id': 'sets'},
    {'question': 'What does an elif clause do in an if statement?', 'doc_id': 'if-statements'},
    {'question': 'What values does range provide in a for loop?', 'doc_id': 'for-loops'},
    {'question': 'What does a function return if it has no return statement?', 'doc_id': 'functions'},
    {'question': 'When are default parameter values evaluated?', 'doc_id': 'function-parameters'},
    {'question': 'Why should I specify UTF-8 when opening a text file?', 'doc_id': 'file-formats'},
    {'question': 'How can an f-string control the number of decimal places?', 'doc_id': 'formatting'},
    {'question': 'When does a finally clause run?', 'doc_id': 'try-finally'},
    {'question': 'What is the purpose of __init__ in a class?', 'doc_id': 'classes'},
    {'question': 'How can a derived class reuse a base-class method?', 'doc_id': 'inheritance'},
    {'question': 'How does a for loop get values from an iterator?', 'doc_id': 'iterators'},
    {'question': 'What happens when a generator reaches a yield statement?', 'doc_id': 'generators'},
]
assert {item['doc_id'] for item in evaluation_questions} == set(documents_by_id)
assert len(evaluation_questions) == len(documents_by_id)


STOPWORDS = {
    'what', 'when', 'where', 'which', 'who', 'why', 'how', 'does', 'do',
    'is', 'are', 'a', 'an', 'the', 'to', 'in', 'on', 'of', 'for', 'with',
    'and', 'or', 'can', 'i', 'it', 'be', 'called', 'given', 'useful',
}


def evaluate_retriever(search, questions, k=3):
    rows = []
    for item in questions:
        started = time.perf_counter()
        results = search(item['question'], k=k)
        elapsed_ms = (time.perf_counter() - started) * 1000
        ranked_doc_ids = [chunks[index]['doc_id'] for index, _ in results]
        try:
            rank = ranked_doc_ids.index(item['doc_id']) + 1
        except ValueError:
            rank = None
        retrieved_text = ' '.join(chunks[index]['text'] for index, _ in results)
        query_keywords = {
            token for token in tokenize(item['question'])
            if len(token) > 2 and token not in STOPWORDS
        }
        retrieved_keywords = set(tokenize(retrieved_text))
        keyword_coverage = (
            len(query_keywords & retrieved_keywords) / len(query_keywords)
            if query_keywords else 0.0
        )
        rows.append({
            'question': item['question'],
            'expected_doc': item['doc_id'],
            'rank': rank,
            'hit_at_k': int(rank is not None),
            'reciprocal_rank': 1 / rank if rank is not None else 0.0,
            'latency_ms': elapsed_ms,
            'keyword_coverage': keyword_coverage,
        })
    return rows


def summarize_evaluation(rows):
    if not rows:
        raise ValueError('Setul de evaluare nu poate fi gol.')
    return {
        'Hit@k': sum(row['hit_at_k'] for row in rows) / len(rows),
        'MRR': sum(row['reciprocal_rank'] for row in rows) / len(rows),
        'KW@k': sum(row['keyword_coverage'] for row in rows) / len(rows),
        'mean_latency_ms': sum(row['latency_ms'] for row in rows) / len(rows),
    }


In [ ]:
retrievers = {
    'BM25': make_retriever('bm25'),
    'Semantic / NumPy': make_retriever('semantic_numpy'),
    'Semantic / FAISS': make_retriever('semantic_faiss'),
    'Hybrid RRF': make_retriever('rrf'),
    'RRF + dense rerank': make_retriever('rerank'),
}

evaluation_results = {}
for name, search in retrievers.items():
    rows = evaluate_retriever(search, evaluation_questions, k=3)
    evaluation_results[name] = rows
    print(name, summarize_evaluation(rows))

print('\nErori de recuperare pentru Hybrid RRF:')
for row in evaluation_results['Hybrid RRF']:
    if not row['hit_at_k']:
        print('-', row['question'], '→', row['expected_doc'])


In [ ]:
threshold_results = []
THRESHOLDS = (0.20, 0.30, 0.40, 0.50)
K_VALUES = (1, 3, 5)
for threshold in THRESHOLDS:
    rows = evaluate_retriever(
        lambda query, k: semantic_threshold_search(query, k, threshold),
        evaluation_questions,
        k=3,
    )
    threshold_results.append({
        'threshold': threshold,
        **summarize_evaluation(rows),
    })
    print(f'threshold={threshold:.2f}', threshold_results[-1])

k_sweep = {}
for k_value in K_VALUES:
    k_sweep[k_value] = {}
    for name, search in retrievers.items():
        rows = evaluate_retriever(search, evaluation_questions, k=k_value)
        k_sweep[k_value][name] = summarize_evaluation(rows)
    print(f'\nk={k_value}:', k_sweep[k_value])


### 7.1 Tabele, grafice și exportul rezultatelor

Celula următoare generează grafice și CSV-uri numai după ce evaluările au fost rulate. Fișierele nu sunt prepopulate cu rezultate estimate. Păstrează numele modelului, pragul și data rulării în raportul final pentru ca rezultatele să poată fi interpretate.

In [ ]:
results_dir = Path('rag_results')
results_dir.mkdir(parents=True, exist_ok=True)
experiment_config = {
    'author': 'Carp Dan',
    'run_timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'python_version': platform.python_version(),
    'corpus_language': 'English',
    'document_count': len(documents),
    'chunk_count': len(chunks),
    'chunk_size_words': CHUNK_SIZE,
    'overlap_words': CHUNK_OVERLAP,
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'generation_model': 'google/flan-t5-small',
    'retrieval_methods': list(retrievers),
    'k_values': list(K_VALUES),
    'similarity_thresholds': list(THRESHOLDS),
}
with (results_dir / 'experiment_config.json').open('w', encoding='utf-8') as file:
    json.dump(experiment_config, file, ensure_ascii=False, indent=2)

retrieval_detail_rows = []
retrieval_summary_rows = []
for method_name, rows in evaluation_results.items():
    retrieval_summary_rows.append({
        'method': method_name,
        'k': 3,
        **summarize_evaluation(rows),
    })
    retrieval_detail_rows.extend(
        {'method': method_name, **row} for row in rows
    )

sweep_rows = [
    {'k': k_value, 'method': method_name, **metrics}
    for k_value, methods in k_sweep.items()
    for method_name, metrics in methods.items()
]

for filename, rows in (
    ('document_catalog.csv', [
        {key: document[key] for key in ('id', 'title', 'source')}
        for document in documents
    ]),
    ('retrieval_by_question.csv', retrieval_detail_rows),
    ('retrieval_summary.csv', retrieval_summary_rows),
    ('retrieval_k_sweep.csv', sweep_rows),
    ('retrieval_threshold_sweep.csv', threshold_results),
):
    if rows:
        with (results_dir / filename).open('w', encoding='utf-8', newline='') as file:
            writer = csv.DictWriter(file, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)

methods = [row['method'] for row in retrieval_summary_rows]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, metric, title in zip(
    axes,
    ('Hit@k', 'MRR', 'mean_latency_ms'),
    ('Retrieval Hit@3', 'Mean Reciprocal Rank', 'Mean latency (ms)'),
):
    values = [row[metric] for row in retrieval_summary_rows]
    axis.bar(methods, values)
    axis.set_title(title)
    axis.tick_params(axis='x', labelrotation=35)
    axis.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(results_dir / 'retrieval_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(
    [row['threshold'] for row in threshold_results],
    [row['Hit@k'] for row in threshold_results],
    marker='o',
)
axes[0].set_title('Retrieval Hit@3')
axes[0].set_xlabel('Cosine similarity threshold')
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.25)
axes[1].plot(
    [row['threshold'] for row in threshold_results],
    [row['mean_latency_ms'] for row in threshold_results],
    marker='s',
)
axes[1].set_title('Mean latency (ms)')
axes[1].set_xlabel('Cosine similarity threshold')
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(results_dir / 'threshold_tradeoff.png', dpi=160, bbox_inches='tight')
plt.show()


### Întrebări de analiză

- Care metodă are cel mai bun MRR în această evaluare și ce exemple explică diferența?
- Ce se schimbă dacă `k` devine 1 sau 5? Recalculează metricile, nu presupune rezultatul.
- Cât din latența semantică poate proveni din codarea întrebării, iar cât din căutarea în matrice?
- Setul de întrebări este reprezentativ? Propune exemple cu formulări diferite și cu răspunsul în afara corpusului.

Nu copia valori de performanță în raport înainte de a rula celulele în mediul tău.

## 8. Generarea unui răspuns fundamentat

Pentru generare folosim `google/flan-t5-small`, un model compact pentru instrucțiuni în engleză. Corpusul este tot în engleză pentru a reduce nepotrivirea limbii între model și surse. Modelul poate genera afirmații greșite chiar dacă promptul cere să nu o facă; răspunsul se verifică față de fragmentele afișate.

Această celulă descarcă modelul numai când notebook-ul este rulat. Dacă descărcarea nu este posibilă, evaluarea retrieval de mai sus rămâne independentă de modelul generativ.

In [ ]:
generator = pipeline(
    'text2text-generation',
    model='google/flan-t5-small',
    device=-1,
)


In [ ]:
def answer_question(question, k=3, min_similarity=0.30):
    nearest = dense_search(question, k=1)
    if not nearest or nearest[0][1] < min_similarity:
        retrieved = []
    else:
        retrieved = rrf_search(question, k=k)
    context_parts = []
    citations = []
    seen_docs = set()
    for index, _ in retrieved:
        chunk = chunks[index]
        context_parts.append(f"[{chunk['doc_id']}] {chunk['text']}")
        if chunk['doc_id'] not in seen_docs:
            citations.append({
                'id': chunk['doc_id'],
                'title': chunk['title'],
                'url': chunk['source'],
            })
            seen_docs.add(chunk['doc_id'])

    context = '\n'.join(context_parts)
    prompt = (
        'Answer the question using only the supplied context. '
        'If the context does not contain the answer, reply exactly: '
        'Not found in the provided corpus. Cite supporting document IDs '
        'in square brackets.\n\n'
        f'Context:\n{context}\n\nQuestion: {question}\nAnswer:'
    )
    generated = generator(prompt, max_new_tokens=120, do_sample=False)[0]
    return {'answer': generated['generated_text'].strip(), 'citations': citations}


## 9. Generarea: prompt curat versus prompt augmentat

Baseline-ul fără RAG primește întrebarea fără fragmente recuperate; varianta RAG primește contextul și poate furniza documente candidate. Comparam aceleași întrebări, inclusiv una fără răspuns în baza de cunoștințe. Latența este măsurată separat pentru fiecare cale; rezultatele se generează numai la rularea notebook-ului.

In [ ]:
generation_questions = [
    {'id': 'pop', 'question': 'What does list.pop return with no index?', 'answerable': True},
    {'id': 'files', 'question': 'Why is with useful for file handling?', 'answerable': True},
    {'id': 'modules', 'question': 'How do I access a function from an imported module?', 'answerable': True},
    {'id': 'outside-corpus', 'question': 'How do I install a package from PyPI?', 'answerable': False},
]


def answer_without_rag(question):
    prompt = (
        'Answer this Python question briefly. If you are unsure, say so. '
        f'Question: {question}\nAnswer:'
    )
    generated = generator(prompt, max_new_tokens=120, do_sample=False)[0]
    return generated['generated_text'].strip()


def compare_generation(question):
    started = time.perf_counter()
    clean_answer = answer_without_rag(question)
    clean_latency_ms = (time.perf_counter() - started) * 1000

    started = time.perf_counter()
    grounded_result = answer_question(question)
    rag_latency_ms = (time.perf_counter() - started) * 1000
    return {
        'without_rag': {
            'answer': clean_answer,
            'citations': [],
            'latency_ms': clean_latency_ms,
        },
        'with_rag': {
            **grounded_result,
            'latency_ms': rag_latency_ms,
        },
    }


generation_results = []
for item in generation_questions:
    result = compare_generation(item['question'])
    generation_results.append({**item, **result})
    print(f"\n[{item['id']}] {item['question']}")
    print('Fără RAG:', result['without_rag']['answer'])
    print('Cu RAG:', result['with_rag']['answer'])
    print('Surse candidate:', result['with_rag']['citations'])
    print('Latență ms:', {
        'without_rag': round(result['without_rag']['latency_ms'], 2),
        'with_rag': round(result['with_rag']['latency_ms'], 2),
    })


## 10. Evaluarea generării

Acuratețea semantică, abținerea corectă și suportul citărilor necesită judecată față de sursele deschise; nu le calculăm automat prin potrivire de șiruri. După examinarea fiecărui răspuns și a URL-ului, completează valorile booleene de mai jos (`True`/`False`). Nu marca un răspuns drept corect doar fiindcă sună plauzibil.

Rubrică: `answer_correct` înseamnă că răspunsul rezolvă întrebarea fără afirmații materiale nesusținute; `abstained` înseamnă refuz explicit de a răspunde când corpusul nu conține informația; `citation_present` înseamnă că răspunsul cu RAG include un ID de sursă; `citation_supported` înseamnă că sursa citată susține afirmația asociată. Pentru răspunsurile fără RAG, criteriile de citare nu se aplică și sunt ignorate în sumar.

In [ ]:
# Completează un rând pentru fiecare întrebare și sistem, după verificarea manuală.
# Câmpuri: system ('without_rag' sau 'with_rag'), question_id,
# answer_correct, abstained, citation_present, citation_supported.
generation_judgments = []


def summarize_generation_judgments(judgments, results):
    if not judgments:
        raise ValueError('Completează mai întâi generation_judgments după verificarea manuală.')
    required = {
        'system', 'question_id', 'answer_correct', 'abstained',
        'citation_present', 'citation_supported',
    }
    allowed_systems = {'without_rag', 'with_rag'}
    question_by_id = {item['id']: item for item in generation_questions}
    expected_pairs = {
        (system, question_id)
        for system in allowed_systems
        for question_id in question_by_id
    }
    observed_pairs = []
    for judgment in judgments:
        if required - judgment.keys():
            raise ValueError(f'Câmpuri lipsă în evaluare: {required - judgment.keys()}')
        if judgment['system'] not in allowed_systems:
            raise ValueError(f"Sistem necunoscut: {judgment['system']}")
        if judgment['question_id'] not in question_by_id:
            raise ValueError(f"Întrebare necunoscută: {judgment['question_id']}")
        boolean_fields = required - {'system', 'question_id'}
        if any(not isinstance(judgment[key], bool) for key in boolean_fields):
            raise ValueError('Completează fiecare criteriu cu True sau False; pentru citări fără RAG, folosește False și nu interpreta acele scoruri.')
        observed_pairs.append((judgment['system'], judgment['question_id']))

    if len(observed_pairs) != len(set(observed_pairs)):
        raise ValueError('Există evaluări duplicate pentru același sistem și aceeași întrebare.')
    missing_pairs = expected_pairs - set(observed_pairs)
    if missing_pairs:
        raise ValueError(f'Lipsesc evaluări pentru combinațiile: {sorted(missing_pairs)}')

    summaries = {}
    for system in ('without_rag', 'with_rag'):
        system_rows = [row for row in judgments if row['system'] == system]
        answerable = [row for row in system_rows if question_by_id[row['question_id']]['answerable']]
        unanswerable = [row for row in system_rows if not question_by_id[row['question_id']]['answerable']]
        result_by_id = {row['id']: row[system] for row in results}
        summaries[system] = {
            'answer_accuracy': sum(row['answer_correct'] for row in answerable) / max(len(answerable), 1),
            'correct_abstention_rate': sum(row['abstained'] for row in unanswerable) / max(len(unanswerable), 1),
            'citation_coverage': (
                sum(row['citation_present'] for row in answerable) / max(len(answerable), 1)
                if system == 'with_rag' else None
            ),
            'supported_citation_rate': (
                sum(row['citation_supported'] for row in answerable) / max(len(answerable), 1)
                if system == 'with_rag' else None
            ),
            'mean_latency_ms': sum(result_by_id[row['question_id']]['latency_ms'] for row in system_rows) / len(system_rows),
            'n_judged': len(system_rows),
        }
    return summaries


# După completarea generation_judgments:
# generation_metrics = summarize_generation_judgments(generation_judgments, generation_results)
# print(generation_metrics)


Pentru fiecare întrebare, compară răspunsul fără RAG cu cel cu RAG. Notează separat dacă răspunsul este corect, dacă modelul se abține pentru întrebarea din afara corpusului, dacă include citare și dacă sursa chiar susține afirmația. Rata de abținere de mai sus este calculată pe întrebările neacoperite; completează un rând pentru fiecare combinație între sistem și întrebare.

In [ ]:
if generation_judgments:
    generation_metrics = summarize_generation_judgments(
        generation_judgments, generation_results
    )
    results_dir.mkdir(parents=True, exist_ok=True)
    print('Generation metrics:', generation_metrics)

    generation_detail_rows = []
    for result in generation_results:
        for system in ('without_rag', 'with_rag'):
            system_result = result[system]
            generation_detail_rows.append({
                'question_id': result['id'],
                'question': result['question'],
                'answerable': result['answerable'],
                'system': system,
                'answer': system_result['answer'],
                'citation_ids': '; '.join(
                    citation['id'] for citation in system_result['citations']
                ),
                'latency_ms': system_result['latency_ms'],
            })

    with (results_dir / 'generation_comparison.csv').open(
        'w', encoding='utf-8', newline=''
    ) as file:
        writer = csv.DictWriter(file, fieldnames=list(generation_detail_rows[0]))
        writer.writeheader()
        writer.writerows(generation_detail_rows)

    with (results_dir / 'generation_judgments.csv').open(
        'w', encoding='utf-8', newline=''
    ) as file:
        writer = csv.DictWriter(file, fieldnames=list(generation_judgments[0]))
        writer.writeheader()
        writer.writerows(generation_judgments)

    metric_rows = [
        {'system': system, **metrics}
        for system, metrics in generation_metrics.items()
    ]
    with (results_dir / 'generation_metrics.csv').open(
        'w', encoding='utf-8', newline=''
    ) as file:
        writer = csv.DictWriter(file, fieldnames=list(metric_rows[0]))
        writer.writeheader()
        writer.writerows(metric_rows)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    systems = list(generation_metrics)
    axes[0].bar(
        systems,
        [generation_metrics[system]['answer_accuracy'] for system in systems],
    )
    axes[0].set_title('Answer accuracy on answerable questions')
    axes[1].bar(
        systems,
        [generation_metrics[system]['correct_abstention_rate'] for system in systems],
    )
    axes[1].set_title('Correct abstention on unanswerable questions')
    axes[2].bar(
        systems,
        [generation_metrics[system]['mean_latency_ms'] for system in systems],
    )
    axes[2].set_title('Mean generation latency (ms)')
    for axis in axes:
        if axis is not axes[2]:
            axis.set_ylim(0, 1)
        axis.grid(axis='y', alpha=0.25)
        axis.tick_params(axis='x', labelrotation=15)
    fig.tight_layout()
    fig.savefig(results_dir / 'generation_quality.png', dpi=160, bbox_inches='tight')
    plt.show()
else:
    print('Add and verify all manual judgments to export generation metrics and plots.')


### Exemplu inspectabil de răspuns RAG


In [ ]:
response = answer_question('Why should I use with when reading a file?')
print(response['answer'])
print('\nSurse recuperate:')
for citation in response['citations']:
    print(f"[{citation['id']}] {citation['title']} — {citation['url']}")


### Verificare manuală a generării

Înregistrează observațiile după rulare; nu completa tabelul cu rezultate anticipate.

| Întrebare | Afirmații susținute de context? | Citare prezentă și corectă? | Abținere dacă lipsește răspunsul? |
|---|---|---|---|
| Întrebare din corpus | de completat | de completat | nu se aplică |
| Întrebare fără răspuns în corpus | de completat | de completat | de completat |

Încearcă, de exemplu, o întrebare despre instalarea unui pachet sau configurarea unui mediu virtual. Aceste informații nu sunt incluse în corpus și răspunsul potrivit este abținerea, nu completarea din cunoștințele generale ale modelului.

## 11. Asistentul final

Funcția de mai jos leagă recuperarea, generarea și afișarea surselor. Într-o aplicație reală, sursele ar trebui afișate lângă afirmațiile pe care le susțin, iar corpusul și modelul ar trebui versionate.

In [ ]:
def ask(question):
    result = answer_question(question)
    print('Răspuns:', result['answer'])
    print('Surse candidate:')
    for citation in result['citations']:
        print(f"- [{citation['id']}] {citation['title']}: {citation['url']}")
    return result


# Exemplu de utilizare după rularea celulelor anterioare:
# ask('How can I handle an exception while working with a file?')


## 12. Concluzii și limite

Completează după experimente:

1. Ce metodă a recuperat cel mai bine sursa pentru întrebările tale și cum ai verificat asta?
2. Ce întrebare a produs cel mai slab rezultat și ce ai schimba: corpusul, segmentarea sau metoda de căutare?
3. A respectat modelul cerința de abținere? Arată un exemplu și verifică-l față de corpus.
4. Ce afirmații din răspunsul generat sunt susținute de URL-urile citate?

Limite cunoscute: corpusul conține numai 19 rezumate scurte, setul de evaluare este mic și construit manual, iar modelul generativ poate ignora instrucțiunile. KW@k este o măsură euristică bazată pe cuvintele întrebării, nu o metrică standard de relevanță. Răspunsurile nu înlocuiesc documentația oficială; verifică sursa și versiunea relevantă pentru problema concretă.